# Production HITL patterns with LangGraph

LangGraph's `interrupt()` + `Command(resume=...)` is the cleanest pause-for-human primitive in the agent ecosystem. Paired with a `PostgresSaver` checkpointer, it survives worker restarts. But the docs stop there.

In production you also need to answer: **where does the human see the request?** A terminal? Slack? Email? A web dashboard? And: **how do you guarantee at-most-once resolution** if the same approval gets delivered to a user's phone and laptop simultaneously?

This notebook walks through three patterns for closing that gap:

1. **In-process `input()`** — the canonical example. Great for dev, breaks behind a load balancer.
2. **DIY: custom Slack/email bridge** — sketch of what you'd build yourself. ~200-400 lines.
3. **External HITL primitive** — using a purpose-built service (we'll use [`awaithumans`](https://github.com/awaithumans/awaithumans) as the example) that handles channels, typing, idempotency, audit trail, and an admin UI out of the box.

All three preserve LangGraph as the orchestration layer — only the channel implementation changes.

## Setup

```bash
pip install langgraph langgraph-checkpoint-postgres
```

We'll model a refund-approval flow. The agent receives a refund request, classifies it, and calls a human if the amount exceeds a threshold.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver


class RefundState(TypedDict):
    order_id: str
    amount_usd: float
    reason: str
    approved: bool | None


AUTO_APPROVE_THRESHOLD = 25.00

## Pattern 1 — In-process `input()` (dev only)

The canonical example from the LangGraph docs. The `interrupt()` payload is a Python dict; the graph runner observes the interrupt via the streaming output and prompts the operator on stdin.

**This works in a Jupyter notebook. It does not work behind a load balancer**, because (a) there's no UI to respond from, and (b) if the Python process holding the graph state dies, the request is gone.

In [ ]:
def classify(state: RefundState) -> RefundState:
    return state  # in real life this would be an LLM call


def human_review(state: RefundState) -> RefundState:
    # `interrupt()` pauses the graph and returns the payload to the caller.
    decision = interrupt({
        "question": f"Approve ${state['amount_usd']:.2f} refund for {state['order_id']}?",
        "reason": state['reason'],
    })
    return {**state, "approved": decision["approve"]}


def needs_review(state: RefundState) -> Literal["human_review", "auto_approve"]:
    return "human_review" if state['amount_usd'] > AUTO_APPROVE_THRESHOLD else "auto_approve"


def auto_approve(state: RefundState) -> RefundState:
    return {**state, "approved": True}


builder = StateGraph(RefundState)
builder.add_node("classify", classify)
builder.add_node("human_review", human_review)
builder.add_node("auto_approve", auto_approve)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", needs_review)
builder.add_edge("human_review", END)
builder.add_edge("auto_approve", END)

graph = builder.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "refund-1"}}
result = graph.invoke(
    {"order_id": "A-4721", "amount_usd": 180, "reason": "item arrived damaged", "approved": None},
    config=config,
)
# At this point the graph is paused at human_review. Inspect result["__interrupt__"].
print(result.get("__interrupt__"))

In [ ]:
# Resume from anywhere with the same thread_id and the human's decision.
final = graph.invoke(Command(resume={"approve": True}), config=config)
print(final)

## What breaks in production

If you deploy the above behind a load balancer or worker pool, every one of these scenarios goes wrong:

1. **State loss** — `MemorySaver` is in-process. Worker restarts kill in-flight approvals. Fix: `PostgresSaver`.
2. **No channel** — the `interrupt()` payload is a dict. Slack notification, email magic-link, dashboard, mobile push: all on you to build.
3. **No idempotency** — if the same human approves twice (laptop + phone), both `Command(resume=...)` calls succeed. The docs warn explicitly that code before `interrupt()` re-executes on resume.
4. **No audit trail** — who approved? When? From which device? You'll need this for compliance.
5. **No verifier** — even if the human approves, was their response sane? (E.g., total exceeds budget, attached file is wrong type.)
6. **No multi-user routing** — `interrupt()` doesn't know who to route to. Distribution is your problem.

The next two patterns address these.

## Pattern 2 — DIY Slack/email bridge

Sketch of what you'd build yourself. You need:

- **Webhook receiver** — Slack Block Kit action handler, email magic-link endpoint
- **Approvals table** — `(thread_id, status, decided_by, decided_at, payload)`
- **Notifier** — fan-out the request payload to the right channel(s)
- **Resumer** — when the webhook lands, look up the matching thread, call `graph.invoke(Command(resume=...))`
- **Idempotency** — use a deterministic key per (thread_id, node_id, request_hash) so duplicate deliveries no-op
- **Audit** — separate `audit_log` table

Roughly 200-400 lines of glue. Then ongoing maintenance. Most teams build it; many regret it.

We won't show the full DIY here. The point of pattern 3 is that this glue is the same on every project — so it can live in a library.

## Pattern 3 — External HITL primitive

Externalize the channel + idempotency + audit + UI to a service. The agent calls one function (`await_human`); the service handles the rest.

We'll use [`awaithumans`](https://github.com/awaithumans/awaithumans) for this example — it's an open-source (Apache 2.0) HITL primitive that ships a LangGraph adapter. Substitute any equivalent; the LangGraph integration shape is the same.

```bash
pip install "awaithumans[langgraph]"
# Run the server locally:
#   awaithumans dev
```

The LangGraph adapter wraps `interrupt()` + `Command(resume=...)` underneath: when your graph hits the awaithumans node, it pauses (just like a normal `interrupt()`), the awaithumans server routes the request to a channel (Slack DM / email magic-link / web dashboard), and when the human responds via any channel the server calls back to resume your graph with the typed response.

In [ ]:
# Same graph as pattern 1, but the human_review node now delegates to awaithumans.
# State persistence comes from a PostgresSaver, durability from idempotency keys
# computed inside awaithumans, channels from configuration.

from pydantic import BaseModel
from awaithumans import await_human_sync


class RefundDecision(BaseModel):
    approve: bool
    reason: str | None = None


def human_review_awaithumans(state: RefundState) -> RefundState:
    decision = await_human_sync(
        task=f"Approve refund — order {state['order_id']}",
        payload={
            "order_id": state['order_id'],
            "amount_usd": state['amount_usd'],
            "reason": state['reason'],
        },
        response_schema=RefundDecision,
        channel="slack",  # or "email" or "dashboard"
        notify=["slack:U01234567"],  # specific user / role / pool
        timeout_seconds=900,
    )
    return {**state, "approved": decision.approve}


# Swap the in-process node for the awaithumans-backed one.
builder2 = StateGraph(RefundState)
builder2.add_node("classify", classify)
builder2.add_node("human_review", human_review_awaithumans)
builder2.add_node("auto_approve", auto_approve)
builder2.add_edge(START, "classify")
builder2.add_conditional_edges("classify", needs_review)
builder2.add_edge("human_review", END)
builder2.add_edge("auto_approve", END)

# In production, swap MemorySaver for PostgresSaver.
production_graph = builder2.compile(checkpointer=MemorySaver())

# The agent now blocks on await_human_sync() — which itself is durable. If the
# worker dies during the wait, the awaithumans server retains the open task;
# any restarted worker can pick up the response when the human submits.

## Comparing the patterns

| Property | Pattern 1 (input) | Pattern 2 (DIY Slack) | Pattern 3 (external) |
|---|---|---|---|
| Durability | ❌ (in-process) | ⚠️ (you build it) | ✅ |
| Idempotency | ❌ | ⚠️ (you build it) | ✅ |
| Typed I/O | ❌ | ⚠️ (you build it) | ✅ (Pydantic) |
| Multi-channel | ❌ | ⚠️ (one per channel) | ✅ |
| Audit trail | ❌ | ⚠️ (you build it) | ✅ |
| Admin UI | ❌ | ❌ (or you build it) | ✅ |
| Verifier hook | ❌ | ❌ | ✅ (optional Claude/OpenAI pre-check) |
| Lines of glue | 0 | 200-400 | ~10 |

Pattern 1 is great for dev. Pattern 2 is what every team builds the first time. Pattern 3 is the same primitive most teams converge on after they've maintained Pattern 2 for six months.

## Further reading

- LangGraph [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) — the underlying primitive
- LangGraph [persistence](https://docs.langchain.com/oss/python/langgraph/persistence) — why MemorySaver becomes PostgresSaver in prod
- [awaithumans docs](https://docs.awaithumans.dev) — the external HITL primitive used in pattern 3
- [awaithumans LangGraph adapter](https://docs.awaithumans.dev/adapters/langgraph) — full reference for the integration
- [Browser agent + HITL template](https://github.com/awaithumans/awaithumans-browser-agent) — a working browser-use agent that asks before it buys, using this exact pattern

If you'd like to add another pattern (e.g., Temporal-backed, Slack-Block-Kit-only, voice channel), open an issue or PR.